In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 72
import numpy as np
import sys

import torch
from torch import nn, optim
import torchvision

print(f'PyTorch version= {torch.__version__}')
print(f'torchvision version= {torchvision.__version__}')
print(f'CUDA available= {torch.cuda.is_available()}')

# set the GPU to device 0
Device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():  # CUDNN Installation
    print(f'Available devices: {torch.cuda.device_count()}, Name: {torch.cuda.get_device_name(0)}')

PyTorch version= 2.10.0
torchvision version= 0.25.0
CUDA available= False


In [7]:
from torch.utils.data import DataLoader, WeightedRandomSampler
import torchvision.datasets as dset
import torchvision.transforms as vtransforms

IMG_CHANNEL= 1  # based on the dataset
IMG_SIZE= 32  # image size, square

BATCH_SIZE= 3096

N_REPEAT= 2  # per textbook, 5 for flowers dataset

# found the following from studies
means_flowers102, stdevs_flowers102 = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
means_mnist, stdevs_mnist = (0.1307), (0.3081)

# use the correct one
# means_data, stdevs_data = means_flowers102, stdevs_flowers102
means_data, stdevs_data = means_mnist, stdevs_mnist

# dif_ds_ = dset.Flowers102(root = "/EP_datasets/flowers102", download=False, split='train',
dif_ds_ = dset.MNIST(root = "EP_datasets/mnist", download=False, train=True,
                     transform=vtransforms.Compose([
                       vtransforms.Resize([IMG_SIZE,IMG_SIZE], antialias=True),
                       vtransforms.ToTensor(),
                       vtransforms.Normalize(means_data, stdevs_data) ]))

sampler = WeightedRandomSampler(torch.ones(len(dif_ds_)), num_samples=N_REPEAT*len(dif_ds_), replacement=True)

# returns data points [0] and labels [1]
Dloader_dif = DataLoader(dif_ds_, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, num_workers=2)

In [4]:
class SinusoidalEmbedding(nn.Module):
    def __init__(self, embed_d=32, max_freq=1000.0):
        from math import log
        super().__init__()
        self.embed_d = embed_d
        self.freqs = torch.exp(torch.linspace(log(1.0), log(max_freq), embed_d//2))
    def forward(self, noise_var):  # noise_var scalar B many
        from math import pi
        angles = 2.0*pi*self.freqs.to(noise_var.device)*noise_var.unsqueeze(-1)  # B x embed//2
        sin_emb = torch.sin(angles)  # B x embed//2
        cos_emb = torch.cos(angles)  # B x embed//2
        # B x embed//2 x 1 x 1
        return torch.cat([sin_emb, cos_emb], dim=-1).unsqueeze(-1).unsqueeze(-1)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding='same')
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding='same')
        self.norm1 = nn.BatchNorm2d(out_ch)
        self.norm2 = nn.BatchNorm2d(out_ch)
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        from torch.nn.functional import silu
        residual = self.residual(x)
        x = silu(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return silu(x + residual)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, block_depth=2):
        super().__init__()
        self.blocks = nn.ModuleList(
            [ResidualBlock(in_ch if i==0 else out_ch, out_ch) for i in range(block_depth)])
        self.pool = nn.AvgPool2d(2)
    def forward(self, x, skips):
        for block in self.blocks:
            x = block(x)
            skips.append(x)  # push and store skips
        return self.pool(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, block_depth=2):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.blocks = nn.ModuleList(
            [ResidualBlock(in_ch+out_ch, out_ch)] +
            [ResidualBlock(out_ch+out_ch, out_ch) for _ in range(block_depth-1)])
    def forward(self, x, skips):
        x = self.up(x)
        for block in self.blocks:
            skip = skips.pop()  # retrieve skips
            x = torch.cat([x, skip], dim=1)
            x = block(x)
        return x


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.initial_conv = nn.Conv2d(IMG_CHANNEL, 32, kernel_size=1)
        self.noise_embedding = SinusoidalEmbedding(embed_d=32)  # 32 size
        self.upsample = nn.Upsample(size=(IMG_SIZE, IMG_SIZE), mode='nearest')

        self.down1 = DownBlock(32+32, 32)  # initial conv + noise_embedding embed_d
        self.down2 = DownBlock(32, 64)
        self.down3 = DownBlock(64, 96)

        self.res1 = ResidualBlock(96, 128)
        self.res2 = ResidualBlock(128, 128)

        self.up1 = UpBlock(128, 96)
        self.up2 = UpBlock(96, 64)
        self.up3 = UpBlock(64, 32)

        self.final_conv = nn.Conv2d(32, IMG_CHANNEL, kernel_size=1)
        nn.init.kaiming_normal_(self.final_conv.weight, nonlinearity='linear')

    def forward(self, noisy_images, noise_var):
        x = self.initial_conv(noisy_images)
        noise_embed = self.noise_embedding(noise_var)
        noise_embed = self.upsample(noise_embed)

        x = torch.cat([x, noise_embed], dim=1)  # textbook pg.217
        skips = []  # stack data structure

        x = self.down1(x, skips)
        x = self.down2(x, skips)
        x = self.down3(x, skips)

        x = self.res1(x)
        x = self.res2(x)

        x = self.up1(x, skips)
        x = self.up2(x, skips)
        x = self.up3(x, skips)

        return self.final_conv(x)

In [5]:
def cos_dif_schedule(dif_t):
    from math import pi
    return torch.cos(dif_t*pi/2), torch.sin(dif_t*pi/2)  # alpha_t, sigma_t

In [6]:
def train(_model, epochs=50, debug=False, fnsave=f'nn_dif_model.pth'):
    loss_tr = []
    optimizer = optim.AdamW(_model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()
    for e in range(epochs):
        loss_e = []
        for i_, (x_0, _) in enumerate(Dloader_dif):
            x_0 = x_0.to(Device)

            noise = torch.randn_like(x_0)
            dif_times = torch.rand((x_0.shape[0],), dtype=torch.float32, device=Device)  # scalar B many
            noise_rates, signal_rates = cos_dif_schedule(dif_times)
            noisy_images = (signal_rates.view(BATCH_SIZE,1,1,1)*x_0 +
                            noise_rates.view(BATCH_SIZE,1,1,1)*noise)

            pred_noise = _model(noisy_images, noise_rates**2)

            # weight the loss wrt to the noise amount
            loss = criterion(pred_noise, noise) * noise_rates.view(BATCH_SIZE,1,1,1).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if i_ == 0 and debug:  # debug
                torch.save(noisy_images.cpu(), f"noisy_img_{e}.pt")
                torch.save(pred_noise.cpu(), f"pred_noise_{e}.pt")
            sys.stderr.write(f"\r{e+1:03d}/{epochs:3d} | loss: {loss.item():6.3f}")
            sys.stderr.flush()
            loss_e += [loss.item()]
        loss_tr += [(np.array(loss_e).mean(), np.array(loss_e).std())]
    torch.save(_model.state_dict(), fnsave)
    return loss_tr

In [ ]:
%%time

model_unet = UNet().to(Device)  # GPU
Loss_tr = train(model_unet, epochs=30, fnsave=f'nn_dif_mnist_.pth')

In [ ]:
def plot_loss(_loss):
    # plot the error, skip first error due to large difference, initial condition
    fig, ax = plt.subplots(1, figsize=(5, 4))
    x = [_ for _ in range(len(_loss[1:]))]
    y1_, y2_ = np.array([_[0] for _ in _loss[1:]]), np.array([_[1] for _ in _loss[1:]])
    ax.plot(y1_, label='Training Loss', c='coral')
    ax.fill_between(x, y1_-y2_, y1_+y2_, color='coral', alpha=0.1)
    ax.set(xlabel='Epochs', ylabel='Loss')
    plt.grid(axis='y')
    plt.legend()
    plt.show()

In [9]:
%%time

Device = torch.device("cpu")  # for inference or generation use CPU, just demonstration
model_unet = UNet().to(Device)
model_unet.load_state_dict(torch.load(f'nn_dif_mnist_.pth', weights_only=True))

CPU times: user 15.7 ms, sys: 3.41 ms, total: 19.1 ms
Wall time: 17.1 ms


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
N_SAMPLES = 8
DIF_STEPS = 5

def plot(_s):
    fig, ax = plt.subplots(N_SAMPLES//8, 8, figsize=(8, N_SAMPLES//8))
    _s = _s.permute(0,2,3,1).numpy()
    for i in range(N_SAMPLES):
        ax.flat[i].imshow(_s[i])
        ax.flat[i].axis("off")
    plt.show()

In [ ]:
torder = (1,1,1,1) if IMG_CHANNEL == 1 else (1,3,1,1)  # tensor order to be able to use plt.imshow()
means = torch.tensor(means_data, device=Device).view(*torder)
stdevs = torch.tensor(stdevs_data, device=Device).view(*torder)

model_unet.eval()
with torch.no_grad():
    current_images = torch.randn((N_SAMPLES, IMG_CHANNEL, IMG_SIZE, IMG_SIZE), device=Device)  # random noise
    plot(torch.clamp(current_images, 0, 1))

    noise_rates = torch.FloatTensor([0.75]*N_SAMPLES).to(Device)
    pred_images = current_images

    for step in range(DIF_STEPS):
        pred_noises = model_unet(pred_images, noise_rates**2)
        pred_images = torch.clamp(pred_images-pred_noises, 0, 1)
        noise_rates *= 0.80
        plot(pred_images*stdevs+means)